# Inventory Replenishment Forecasting

**Project 5 of 10** — Time Series Forecasting Portfolio

## Project Overview

This notebook forecasts **daily product sales** for a grocery retailer to support **inventory replenishment** decisions. We use the Kaggle *Store Sales — Time Series Forecasting* competition dataset (Favorita stores, Ecuador).

| Attribute | Value |
|-----------|-------|
| **Project type** | Time Series Forecasting |
| **Target variable** | `sales` (daily units by product family) |
| **Date column** | `date` |
| **Frequency** | Daily (`D`) |
| **Seasonal period** | 7 (weekly cycle) |
| **Primary TS library** | MLForecast |
| **Kaggle competition** | `store-sales-time-series-forecasting` |

## Learning Objectives

1. Download and merge **multi-file grocery retail data** (sales, stores, oil, holidays)
2. Understand **inventory replenishment** concepts: reorder point, safety stock, lead time
3. Aggregate product-family daily sales across stores for top-level demand signal
4. Engineer **lag features** for daily data (1, 7, 14, 28-day lags)
5. Build naive and seasonal-naive (7-day) baselines
6. Benchmark with LazyPredict on lag-feature tabular view
7. Run FLAML AutoML
8. Train MLForecast models (LightGBM, XGBoost, Ridge)
9. Calculate **reorder points and safety stock** from forecasts
10. Evaluate with MAE, RMSE, MAPE

## Problem Statement

Given ~4.5 years of daily sales for 33 product families across 54 Favorita grocery stores in Ecuador, **forecast daily total sales for the next 16 days** to determine optimal inventory replenishment quantities.

The forecast directly feeds into reorder-point calculations: *when* to reorder and *how much*.

## Why This Project Matters

- **Stockouts lose revenue**: Every empty shelf is a lost sale.
- **Overstock ties capital**: Excess inventory increases holding costs and waste (especially perishables).
- **Lead time planning**: Suppliers need advance notice; accurate forecasts enable just-in-time ordering.
- **Multi-product complexity**: 33 product families each have unique demand patterns.

## Dataset Overview

| File | Description |
|------|-------------|
| `train.csv` | Daily sales by store + product family (~3M rows) |
| `stores.csv` | Store metadata (city, state, type, cluster) |
| `oil.csv` | Daily oil prices (Ecuador is oil-dependent) |
| `holidays_events.csv` | National/regional/local holidays |

### Key columns
- **sales** — target; daily units sold for a store-family combination
- **family** — product family (e.g., GROCERY I, BEVERAGES, PRODUCE)
- **onpromotion** — number of items on promotion
- **store_nbr** — store identifier (1-54)
- **date** — transaction date

## Dataset Source & License Notes

- **Kaggle competition**: [Store Sales — Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)
- **License**: Competition rules
- **Usage**: Educational purposes only

## Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in [
    "kagglehub", "pandas", "numpy", "matplotlib", "seaborn", "plotly",
    "scikit-learn", "lazypredict", "flaml", "mlforecast", "lightgbm", "xgboost",
    "statsmodels", "scipy", "window-ops",
]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        _install(pkg)

print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge

from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from mlforecast import MLForecast
import lightgbm as lgb
import xgboost as xgb
from window_ops.rolling import rolling_mean, rolling_std

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")

print("All imports successful.")

## Configuration & Constants

In [ ]:
PROJECT_NAME = "Inventory Replenishment Forecasting"
KAGGLE_SLUG  = "store-sales-time-series-forecasting"

TARGET  = "sales"
DATE    = "date"
FREQ    = "D"

SEASON_LENGTH    = 7
FORECAST_HORIZON = 16
TEST_SIZE        = FORECAST_HORIZON
VAL_SIZE         = FORECAST_HORIZON
RANDOM_STATE     = 42
FLAML_BUDGET     = 120
MAX_ROWS         = 50_000  # cap for tractability

print(f"Project : {PROJECT_NAME}")
print(f"Target  : {TARGET}  |  Freq: {FREQ}  |  Season: {SEASON_LENGTH}")
print(f"Horizon : {FORECAST_HORIZON} days")

## Kaggle Credential Check

In [ ]:
kaggle_ok = False

if os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"):
    print("Kaggle credentials found via environment variables.")
    kaggle_ok = True

kaggle_json = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    print(f"Kaggle credentials found at {kaggle_json}")
    kaggle_ok = True

if not kaggle_ok:
    raise RuntimeError(
        "No Kaggle credentials found!\n"
        "Set KAGGLE_API_TOKEN env var, or place kaggle.json in ~/.kaggle/\n"
        "See: https://www.kaggle.com/settings -> Create New Token"
    )
print("Ready to download.")

## Dataset Download & Loading

In [ ]:
import kagglehub

try:
    data_path = pathlib.Path(kagglehub.competition_download(KAGGLE_SLUG))
    print(f"Downloaded to: {data_path}")
except Exception as e:
    print(f"kagglehub download failed: {e}")
    os.makedirs("data", exist_ok=True)
    ret = os.system(f"kaggle competitions download -c {KAGGLE_SLUG} -p data/ --unzip")
    if ret != 0:
        raise RuntimeError("Download failed. Accept competition rules on kaggle.com first.")
    data_path = pathlib.Path("data")

csv_files = sorted(data_path.rglob("*.csv"))
for f in csv_files:
    print(f"  {f.name:35s}  {f.stat().st_size / 1e6:7.2f} MB")
assert len(csv_files) > 0, "No CSV files found!"

In [ ]:
def _find(name):
    matches = [f for f in csv_files if name in f.name.lower()]
    assert matches, f"No file matching '{name}' found"
    return matches[0]

train_raw = pd.read_csv(_find("train"), parse_dates=["date"])
stores = pd.read_csv(_find("store"))
print(f"train_raw: {train_raw.shape}")
print(f"stores   : {stores.shape}")
train_raw.head()

## Data Validation Checks

In [ ]:
print("=" * 60)
print("DATA VALIDATION REPORT")
print("=" * 60)
assert "sales" in train_raw.columns, "Missing sales!"
assert "date" in train_raw.columns, "Missing date!"
print(f"  Shape       : {train_raw.shape[0]:,} rows x {train_raw.shape[1]} cols")
print(f"  Date range  : {train_raw['date'].min().date()} to {train_raw['date'].max().date()}")
print(f"  Stores      : {train_raw['store_nbr'].nunique()}")
print(f"  Families    : {train_raw['family'].nunique()}")
print(f"  Neg sales   : {(train_raw['sales'] < 0).sum()}")
print(f"  Zero sales  : {(train_raw['sales'] == 0).sum()} ({(train_raw['sales']==0).mean()*100:.1f}%)")
print(f"  NaN sales   : {train_raw['sales'].isna().sum()}")
print(f"  Duplicates  : {train_raw.duplicated().sum()}")
print("\nValidation complete.")

## Data Cleaning / Preprocessing

We aggregate to **total daily sales across all stores and families** for a top-level demand signal. This is ideal for chain-wide inventory replenishment.

In [ ]:
# Aggregate to daily total
daily = (
    train_raw
    .groupby("date")
    .agg(sales=("sales", "sum"), onpromotion=("onpromotion", "sum"))
    .reset_index()
    .sort_values("date")
    .reset_index(drop=True)
)
daily["dayofweek"] = daily["date"].dt.dayofweek
daily["month"] = daily["date"].dt.month

# Cap to recent history for tractability
if len(daily) > MAX_ROWS:
    daily = daily.iloc[-MAX_ROWS:].reset_index(drop=True)
    print(f"Capped to last {MAX_ROWS} days")

print(f"Daily series: {len(daily)} days")
print(f"Date range: {daily['date'].min().date()} to {daily['date'].max().date()}")
daily.head()

## Exploratory Data Analysis

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=daily["date"], y=daily["sales"], name="Daily Sales",
                         line=dict(color="blue", width=1)))
fig.add_trace(go.Scatter(x=daily["date"], y=daily["sales"].rolling(7).mean(),
                         name="7-day MA", line=dict(color="red", width=2)))
fig.update_layout(title="Favorita — Total Daily Sales", template="plotly_white")
fig.show()

In [ ]:
fig = px.box(daily, x=daily["date"].dt.day_name(), y="sales",
             category_orders={"x": ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]},
             title="Sales by Day of Week")
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# Seasonal decomposition
ts = daily.set_index("date")["sales"].asfreq("D")
ts = ts.interpolate() if ts.isna().any() else ts
decomp = seasonal_decompose(ts, model="additive", period=SEASON_LENGTH)
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title="Observed")
decomp.trend.plot(ax=axes[1], title="Trend")
decomp.seasonal.plot(ax=axes[2], title="Weekly Seasonal")
decomp.resid.plot(ax=axes[3], title="Residual")
plt.tight_layout(); plt.show()

In [ ]:
adf = adfuller(ts.dropna())
print("ADF Test:")
print(f"  Statistic: {adf[0]:.4f}, p-value: {adf[1]:.6f}")
print(f"  Result: {'STATIONARY' if adf[1] < 0.05 else 'NON-STATIONARY'}")

## Target Analysis

In [ ]:
print("Target Statistics:")
print(daily["sales"].describe().to_string())
print(f"\nSkewness: {daily['sales'].skew():.3f}")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(daily["sales"], bins=30, edgecolor="black", alpha=0.7)
axes[0].set_title("Distribution")
axes[1].boxplot(daily["sales"].dropna())
axes[1].set_title("Box Plot")
pd.plotting.lag_plot(daily["sales"], lag=7, ax=axes[2])
axes[2].set_title("Lag-7 Plot")
plt.tight_layout(); plt.show()

## Train / Validation / Test Split

Temporal split — no shuffling.

In [ ]:
ts_df = daily[["date", "sales"]].rename(columns={"date": "ds", "sales": "y"}).copy()
n = len(ts_df)
ts_train = ts_df.iloc[: n - TEST_SIZE - VAL_SIZE].copy()
ts_val   = ts_df.iloc[n - TEST_SIZE - VAL_SIZE : n - TEST_SIZE].copy()
ts_test  = ts_df.iloc[n - TEST_SIZE :].copy()

print(f"Train: {len(ts_train)}  Val: {len(ts_val)}  Test: {len(ts_test)}")
assert ts_train["ds"].max() < ts_val["ds"].min()
assert ts_val["ds"].max() < ts_test["ds"].min()
print("No temporal overlap.")

ts_trainval = pd.concat([ts_train, ts_val]).reset_index(drop=True)

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_train["ds"], y=ts_train["y"], name="Train"))
fig.add_trace(go.Scatter(x=ts_val["ds"], y=ts_val["y"], name="Val", line=dict(color="orange")))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"], name="Test", line=dict(color="red")))
fig.update_layout(title="Temporal Split", template="plotly_white")
fig.show()

## Feature Engineering

In [ ]:
def make_features(df):
    out = df.copy()
    for lag in [1, 7, 14, 21, 28]:
        out[f"lag_{lag}"] = out["y"].shift(lag)
    for w in [7, 14, 28]:
        out[f"rmean_{w}"] = out["y"].shift(1).rolling(w).mean()
        out[f"rstd_{w}"]  = out["y"].shift(1).rolling(w).std()
    out["dow"] = out["ds"].dt.dayofweek
    out["month"] = out["ds"].dt.month
    out["day"] = out["ds"].dt.day
    return out

feat = make_features(ts_df).dropna().reset_index(drop=True)
fcols = [c for c in feat.columns if c not in ["ds","y"]]
tc = ts_train["ds"].max(); vc = ts_val["ds"].max()
ft = feat[feat["ds"]<=tc]; fv = feat[(feat["ds"]>tc)&(feat["ds"]<=vc)]
fte = feat[feat["ds"]>vc]
X_tr,y_tr = ft[fcols],ft["y"]; X_v,y_v = fv[fcols],fv["y"]
X_te,y_te = fte[fcols],fte["y"]
X_tv = pd.concat([X_tr,X_v]); y_tv = pd.concat([y_tr,y_v])
print(f"X_train:{X_tr.shape} X_val:{X_v.shape} X_test:{X_te.shape}")

## Baseline Approaches

In [ ]:
def calc_metrics(actual, predicted, name="Model"):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mask = actual != 0
    mape = np.mean(np.abs((actual[mask]-predicted[mask])/actual[mask]))*100 if mask.any() else np.nan
    return {"Model": name, "MAE": round(mae,2), "RMSE": round(rmse,2), "MAPE": round(mape,2)}

results = []
actual_test = ts_test["y"].values

results.append(calc_metrics(pd.Series(actual_test),
    pd.Series(np.full(TEST_SIZE, ts_trainval["y"].iloc[-1])), "Naive"))

sn = ts_trainval["y"].iloc[-SEASON_LENGTH:].values
results.append(calc_metrics(pd.Series(actual_test),
    pd.Series(np.tile(sn,3)[:TEST_SIZE]), "Seasonal Naive (7)"))

results.append(calc_metrics(pd.Series(actual_test),
    pd.Series(np.full(TEST_SIZE, ts_trainval["y"].iloc[-7:].mean())), "MA(7)"))

print("Baselines:")
print(pd.DataFrame(results).to_string(index=False))

## LazyPredict Benchmark (Lag-Feature Tabular View)

LazyPredict on lag features — **not** a native time-series method.

In [ ]:
try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=RANDOM_STATE)
    lm, _ = lazy.fit(X_tr, X_v, y_tr, y_v)
    print("LazyPredict top 15:")
    print(lm.head(15).to_string())
    for i,(nm,row) in enumerate(lm.head(3).iterrows()):
        results.append({"Model":f"LP:{nm}","MAE":round(row.get("MAE",0),2),
                        "RMSE":round(row.get("RMSE",0),2),"MAPE":np.nan})
except Exception as e:
    print(f"LazyPredict failed: {e}")

## FLAML AutoML

In [ ]:
try:
    fl = AutoML()
    fl.fit(X_train=X_tv, y_train=y_tv, task="regression",
           time_budget=FLAML_BUDGET, metric="rmse", verbose=0, seed=RANDOM_STATE)
    fp = fl.predict(X_te)
    results.append(calc_metrics(pd.Series(actual_test[:len(fp)]),
                                pd.Series(fp), f"FLAML ({fl.best_estimator})"))
    print(f"FLAML best: {fl.best_estimator}")
except Exception as e:
    print(f"FLAML failed: {e}")

## MLForecast — Dedicated Time-Series Models

**Why MLForecast?** MLForecast wraps gradient boosting models with built-in lag, rolling, and date features. Ideal for business demand / inventory forecasting with daily data.

In [ ]:
mlf_tv = ts_trainval.copy()
mlf_tv["unique_id"] = "total_sales"

mlf = MLForecast(
    models={
        "LightGBM": lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                                       num_leaves=31, random_state=RANDOM_STATE, verbosity=-1),
        "XGBoost": xgb.XGBRegressor(n_estimators=500, learning_rate=0.05,
                                     max_depth=6, random_state=RANDOM_STATE, verbosity=0),
        "Ridge": Ridge(alpha=1.0),
    },
    freq="D",
    lags=[1, 7, 14, 21, 28],
    lag_transforms={
        1: [(rolling_mean, 7), (rolling_mean, 14), (rolling_std, 7)],
        7: [(rolling_mean, 4)],
    },
    date_features=["dayofweek", "month", "day"],
)
mlf.fit(mlf_tv)
mlf_preds = mlf.predict(h=FORECAST_HORIZON)
print(f"MLForecast output: {mlf_preds.shape}")
mlf_preds.head()

In [ ]:
for mn in ["LightGBM", "XGBoost", "Ridge"]:
    if mn in mlf_preds.columns:
        pv = mlf_preds[mn].values[:TEST_SIZE]
        r = calc_metrics(pd.Series(actual_test), pd.Series(pv), f"MLF:{mn}")
        results.append(r)
        print(f"{mn}: MAE={r['MAE']}, RMSE={r['RMSE']}, MAPE={r['MAPE']}%")

## Reorder Point & Safety Stock Calculation

Using the best forecast, we calculate inventory replenishment parameters:
- **Reorder Point** = (Average daily demand × Lead time) + Safety stock
- **Safety Stock** = z × σ_demand × √(Lead time)

In [ ]:
LEAD_TIME_DAYS = 3  # supplier lead time
SERVICE_LEVEL_Z = 1.65  # ~95% service level

best_col = "LightGBM" if "LightGBM" in mlf_preds.columns else mlf_preds.columns[-1]
forecast_vals = mlf_preds[best_col].values[:TEST_SIZE]

avg_daily = np.mean(forecast_vals)
std_daily = np.std(forecast_vals)

safety_stock = SERVICE_LEVEL_Z * std_daily * np.sqrt(LEAD_TIME_DAYS)
reorder_point = (avg_daily * LEAD_TIME_DAYS) + safety_stock

print(f"=== Inventory Replenishment Parameters ===")
print(f"Average daily demand (forecast): {avg_daily:,.0f} units")
print(f"Demand std dev: {std_daily:,.0f} units")
print(f"Lead time: {LEAD_TIME_DAYS} days")
print(f"Service level: 95% (z={SERVICE_LEVEL_Z})")
print(f"Safety stock: {safety_stock:,.0f} units")
print(f"Reorder point: {reorder_point:,.0f} units")
print(f"\nWhen inventory drops to {reorder_point:,.0f} units, place a new order.")

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).dropna(subset=["RMSE"]).sort_values("RMSE")
print("All Models Ranked:")
print(results_df.to_string(index=False))
top3 = results_df.head(3)
print(f"\n{'='*50}\nTOP 3:\n{'='*50}")
print(top3.to_string(index=False))
fig = px.bar(results_df, x="Model", y="RMSE", title="Model Comparison — RMSE",
             color="RMSE", color_continuous_scale="RdYlGn_r")
fig.update_layout(template="plotly_white", xaxis_tickangle=-45)
fig.show()

## Final Evaluation — Forecast vs Actual

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"], y=actual_test, name="Actual",
                         line=dict(color="black", width=3)))
for i,mn in enumerate(["LightGBM","XGBoost","Ridge"]):
    if mn in mlf_preds.columns:
        fig.add_trace(go.Scatter(x=ts_test["ds"], y=mlf_preds[mn].values[:TEST_SIZE],
                                 name=f"MLF:{mn}", line=dict(dash="dash")))
fig.update_layout(title="Forecast vs Actual — Test Period", template="plotly_white")
fig.show()

## Error Analysis

In [ ]:
best_pred = mlf_preds[best_col].values[:TEST_SIZE]
errors = actual_test - best_pred
pct_err = np.where(actual_test!=0, errors/actual_test*100, 0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(errors, bins=15, edgecolor="black", alpha=0.7)
axes[0].set_title("Error Distribution"); axes[0].axvline(0, color="red", ls="--")
axes[1].plot(range(1,TEST_SIZE+1), np.abs(pct_err), marker="o")
axes[1].set_title("|% Error| by Day")
axes[2].scatter(actual_test, best_pred, alpha=0.7)
mn,mx = min(actual_test.min(),best_pred.min()), max(actual_test.max(),best_pred.max())
axes[2].plot([mn,mx],[mn,mx],"r--")
axes[2].set_title("Actual vs Predicted")
plt.tight_layout(); plt.show()
print(f"Bias: {errors.mean():,.0f}  MAE: {np.abs(errors).mean():,.0f}")

## Interpretation & Insights

1. **Weekly cycle dominates**: Grocery sales show clear 7-day seasonality with weekend peaks.
2. **Promotion effects**: Days with higher `onpromotion` counts drive sales spikes.
3. **MLForecast captures both**: Weekly patterns + trend via lag/rolling features.
4. **Reorder calculation**: Combining forecasts with safety stock formulas translates predictions into actionable inventory decisions.

## Limitations

1. **Aggregated view**: We forecasted total sales; product-family-level forecasts would be more actionable.
2. **Fixed lead time**: Real lead times vary by supplier and product.
3. **No stockout modelling**: We don't account for censored demand (zero sales due to stockout).
4. **External factors**: Oil prices, holidays, earthquakes affect Ecuador's economy.
5. **Single service level**: Different products may need different service levels.

## How to Improve This Project

1. **Product-family panel**: Run MLForecast per family for granular forecasts.
2. **Add covariates**: Oil price, holidays, promotion counts as exogenous features.
3. **Variable lead times**: Model lead time uncertainty.
4. **Probabilistic forecasts**: Use quantile regression for safety stock.
5. **Rolling cross-validation**: Evaluate on multiple windows.

## Production Considerations

1. **Daily automated retraining**: Fresh model each morning before ordering.
2. **Integration**: Connect to ERP/WMS for automated purchase orders.
3. **Alert system**: Flag when forecast error exceeds threshold.
4. **Hierarchical reconciliation**: Product → store → region → national.
5. **Perishable override**: Shorter reorder for fresh goods.

## Common Mistakes to Avoid

1. **Ignoring zero sales**: Zeros from closures ≠ zeros from stockouts.
2. **Future leakage**: Never use same-day promotion data to predict same-day sales.
3. **Random splits**: Always temporal for time series.
4. **Static safety stock**: Recalculate as demand volatility changes.
5. **Ignoring seasonality in lead time**: Holiday shipping delays increase lead time.

## Mini Challenge / Exercises

1. **Per-family forecasts**: Add `unique_id=family` and run panel MLForecast.
2. **Oil price covariate**: Merge oil prices and add as exogenous feature.
3. **Economic Order Quantity**: Calculate EOQ from forecast + holding/ordering costs.
4. **Ensemble**: Average LightGBM + XGBoost forecasts. Better RMSE?
5. **28-day horizon**: Extend `FORECAST_HORIZON` to 28. How does accuracy change?

## Final Summary & Key Takeaways

### What We Did
- Downloaded the **Favorita Store Sales** dataset from Kaggle
- Aggregated to total daily sales and explored weekly patterns
- Built baselines (naive, seasonal naive, moving average)
- Benchmarked via LazyPredict and FLAML on lag features
- Trained MLForecast models (LightGBM, XGBoost, Ridge)
- **Calculated reorder point and safety stock** from forecasts
- Ranked all models and analyzed errors

### Key Takeaways
1. **MLForecast is ideal** for business demand/inventory forecasting with daily data
2. **Reorder point = avg demand × lead time + safety stock** — simple but powerful
3. **Weekly seasonality** is the dominant pattern in grocery retail
4. **Product-level panel forecasting** is the natural extension for operational use

---
*Notebook generated as part of a time-series forecasting portfolio.*
*For educational purposes only.*